### Chatbot And RAG Evaluation
Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
- Answer relevance
- Answer accuracy
- Retrieval quality
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

Chatbot Evaluation

In [19]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [21]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "New Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['4d2e5569-56b1-4410-accf-0e95c266bfbc',
  'dba31717-5df9-4aae-b875-039dd2aa466e',
  '943b4d78-2aa4-4d10-b587-91f5ed2fbe16',
  '64a0dc48-29c2-4df3-99c1-3d15c94fd552',
  '46fd82dc-a171-4892-a385-be13472631f2'],
 'count': 5,
 'as_of': '2026-09-14T08:08:25.976337866Z'}

### LLM as Evaluator

In [22]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

eval_instructions = (
    "You are an expert professor specialized in grading students' answers to questions."
)

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    user_content = f"""
You are grading the following question:

{inputs['question']}

Here is the real answer:

{reference_outputs['answer']}

You are grading the following predicted answer:

{outputs['response']}

Respond with CORRECT or INCORRECT:

Grade:
"""

    response = llm.invoke([
        ("system", eval_instructions),
        ("user", user_content)
    ])

    return response.content.strip().upper() == "CORRECT"

In [23]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

## Run Evals

In [24]:
default_instructions = "Respond to the user's question in a short, concise manner (one short sentence)."

def my_app(
    question: str,
    model: str = "openai/gpt-oss-120b",
    instructions: str = default_instructions
) -> str:

    response = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])

    return response.content

In [27]:
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="openai/gpt-oss-120b"
        )
    }

In [29]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="groq-gpt-oss-120b-chatbot"
)

View the evaluation results for experiment: 'groq-gpt-oss-120b-chatbot-70b65a03' at:
https://smith.langchain.com/o/0df93336-04ff-42df-a6e8-b5f0750cff3d/datasets/3c5f7318-3199-46a6-b2c1-8fc165c5219d/compare?selectedSessions=baa3e3aa-ca7a-4992-84ee-dc789f926a36


